# 02a — Tariff Pricer (owns its calculation)
This notebook IS the tariff method: the CALC cell below computes the premium and registers it. `pricing.py` only connects (`quote`/`api_quote`). Hub `02_pricing.ipynb` and `03_main.ipynb` replay this exact cell via the loader — one source of truth.

In [ ]:
import os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

import pandas as pd
from voltvision import load_template, describe_pricer, resolve_cards, api_quote
from voltvision import io
from voltvision.simulate import simulate_book
REGIME = 'tariff'


In [ ]:
# CALC — owned by this notebook. The tariff method owns ALL its numbers:
# the 2015 schedule tables, the knobs, and the premium formula below.
# Scenarios may override any declared card parameter:
#   "pricing": {"tariff": {"tpo_loading": 1.35}}
import numpy as np
from voltvision import loading
from voltvision.pricing import register_pricer

CARD = {
    'sst':         {'default': 0.08,  'unit': 'fraction',       'note': 'premium tax on the tariff leg'},
    'tpo_sa_pct':  {'default': 0.005, 'unit': 'fraction of SA', 'note': 'TPO SA-linked premium slice'},
    'tpo_loading': {'default': 1.10,  'unit': 'multiplier',     'note': 'TPO fixed loading (no cover loading / NCD)'},
    'risk_step':   {'default': 1.1,   'unit': 'per flag',       'note': 'multiplier for each true risk flag (flood/theft)'},
}

# Schedule of Motor Tariff 2015 — RM rates by region and engine band.
# Band labels must match base_template.json "engine_bands" exactly.
TARIFF_BANDS = [
    "0 to 1,400 cc / EV up to 70 kW",
    "1,401 to 1,650 cc / EV 71 - 100 kW",
    "1,651 - 2,200 cc / EV 101 - 125 kW",
    "2,201 - 3,050 cc / EV 126 - 150 kW",
    "3,051 - 4,100 cc / EV 151 - 200 kW",
    "4,101 - 4,250 cc / EV 201 - 250 kW",
    "4,251 - 4,400 cc / EV 251 - 300 kW",
    "Over 4,400 cc / EV > 300 kW",
]
BASIC_COMP = {
    "Peninsular Malaysia": [273.8, 305.5, 339.1, 372.6, 404.3, 436.0, 469.6, 501.3],
    "East Malaysia (Sabah, Sawarak & Labuan)": [196.2, 220.0, 243.9, 266.5, 290.4, 313.0, 336.9, 359.5],
}
BASIC_TPO = {
    "Peninsular Malaysia": [120.6, 135.0, 151.2, 167.4, 181.8, 196.2, 212.4, 226.8],
    "East Malaysia (Sabah, Sawarak & Labuan)": [67.5, 75.6, 85.2, 93.6, 101.7, 110.1, 118.2, 126.6],
}
PER_EXTRA = {
    "Peninsular Malaysia": 26.0,
    "East Malaysia (Sabah, Sawarak & Labuan)": 20.3,
}


def basic_premium(book):
    """Tariff base premium per policy, before loadings / NCD / risk / tax.

    Comprehensive = first-RM1,000 rate + PER_EXTRA per extra RM1,000 of sum
    assured; TPFT = 75% of Comprehensive; TPO = flat table rate (the SA slice
    is added at pricing time).
    """
    band_index = {band: i for i, band in enumerate(TARIFF_BANDS)}
    unknown = set(book['ENGINE_CAPACITY']) - set(band_index)
    if unknown:
        raise ValueError(f"tariff tables missing bands {sorted(unknown)} — "
                         "update 02a tables or base_template engine_bands")
    region = book['REGION'].values
    band = book['ENGINE_CAPACITY'].values
    comp_rate = np.array([BASIC_COMP[r][band_index[b]] for r, b in zip(region, band)])
    extra_rate = np.array([PER_EXTRA[r] for r in region])
    units = np.ceil(np.maximum(0, book['SUM_ASSURED'].values - 1000) / 1000)
    comp_basic = comp_rate + extra_rate * units
    tpo_rate = np.array([BASIC_TPO[r][band_index[b]] for r, b in zip(region, band)])
    coverage = book['COVERAGE_TYPE'].values
    basic = np.where(coverage == 'Comprehensive', comp_basic,
                     np.where(coverage == 'TPFT', np.round(0.75 * comp_basic, 2), tpo_rate))
    return np.round(basic, 2)


def price_tariff(book, card, cfg, base_cfg=None):
    """Standard method interface: (book, card, cfg, base_cfg) -> + FINAL_PREMIUM_SST."""
    o = book.copy()
    base = basic_premium(o)
    ncd = 1 - o['NCD_LEVEL'].values
    flags = o['FLOOD_RISK'].values.astype(int) + o['THEFT_RISK'].values.astype(int)
    risk = card.risk_step ** flags
    prem = base * loading(o, cfg) * ncd * risk * (1 + card.sst)
    tpo = o['COVERAGE_TYPE'].values == 'TPO'
    prem[tpo] = ((base[tpo] + card.tpo_sa_pct * o['SUM_ASSURED'].values[tpo])
                 * card.tpo_loading * risk[tpo] * (1 + card.sst))
    return o.assign(FINAL_PREMIUM_SST=prem.round(2))


register_pricer('tariff', price_tariff, card=CARD, info={
    'label': 'Tariff',
    'color': '#94a3b8',
    'formula': ('Comp/TPFT = BASIC x loading x (1-NCD) x risk_step^flags x (1+sst); '
                'TPO = (BASIC + tpo_sa_pct x SA) x tpo_loading x risk_step^flags x (1+sst)'),
})
print('tariff calc registered')


## Rule sheet (`describe_pricer('tariff')`)
| Piece | Rule |
|---|---|
| Comp / TPFT | `BASIC × loading × (1−NCD) × risk_step^flags × (1+sst)` |
| TPO | `(BASIC + tpo_sa_pct × SA) × tpo_loading × risk_step^flags × (1+sst)` — no cover loading, no NCD |
| `BASIC` | Schedule of Motor Tariff 2015 computed IN THIS notebook from book attributes: graduated Comp, TPFT = 0.75 × Comp, TPO flat by band/region |
| `loading` | driver band (1.20 / 1.05 / 1.00 / 1.05) × (1 + 0.03 × min(CAR_AGE, 10)) |
| `flags` | FLOOD_RISK + THEFT_RISK count |
| Live params | declared in `CARD` inside the CALC cell (`tpo_loading`, `tpo_sa_pct`, ...) — override per scenario via `"pricing": {"tariff": {...}}` |


In [ ]:
cfg = load_template()
print(describe_pricer(REGIME)['label'], '- declared parameters:')
display(pd.DataFrame(describe_pricer(REGIME)['params']))
cards = resolve_cards(cfg)
print('resolved cards for:', list(cards))


## Request (simulated book in)

In [ ]:
SC, SEED_RUN = 'MIX', 0   # any scenario/seed written by run_scenarios.py
try:
    raw = io.sim_book(io.load_result(SC, SEED_RUN))
    print(f'loaded shared/results/{SC}_s{SEED_RUN}: {len(raw)} rows')
except FileNotFoundError:
    print('no result yet - quick inline sim (run `python run_scenarios.py` for real books)')
    quick = {**load_template(), 'n': 1000, 'n_years': 2}
    raw = simulate_book(quick, quick['vehicle_mix'], seed=0, n_years=2)


## Response (API format: regime + label + card + metrics + priced book)

In [ ]:
resp = api_quote(raw, REGIME, cards[REGIME], cfg)
print('regime:', resp['regime'], '|', resp['label'])
display(pd.DataFrame([resp['metrics']]))
display(resp['book'][['POLID', 'COVERAGE_TYPE', 'VEHICLE_TYPE', 'CLAIM_COUNT',
    'CLAIM_AMOUNT', 'FINAL_PREMIUM_SST']].head())


## Notes
- TPO leg prices below expected cost (TPO LR >100%) — move `tpo_sa_pct` / `tpo_loading` above when approved.
- Siblings: `02b_glm.ipynb`, `02c_telem.ipynb` (same request/response shape).